In [1]:
import sys
sys.path.append('src')

In [2]:
import os

In [3]:
%pwd

'/content'

In [4]:
os.chdir(r'c:\Users\Harsha vardhan\OneDrive\Desktop\TextSummarization-Project')

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\Harsha vardhan\\OneDrive\\Desktop\\TextSummarization-Project'

In [ ]:
%pwd

'c:\\Users\\Harsha vardhan\\OneDrive\\Desktop\\TextSummarization-Project'

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: str
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    eval_steps: int
    save_steps: int
    gradient_accumulation_steps: int

In [ ]:
from textsummarizer.constants import*
from textsummarizer.utils.common import read_yaml, create_directories


In [ ]:
class ConfigurationManager:
    def __init__(self, config_file_path = CONFIG_FILE_PATH,
         params_filepath=PARAMS_FILE_PATH):
        

        self.config= read_yaml(config_file_path)
        self.params= read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config=self.config.model_trainer
        params=self.params.TrainingArgs

        create_directories([config.root_dir])

        model_trainer_config=ModelTrainerConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path),
            model_ckpt=config.model_ckpt,
            num_train_epochs=params.num_train_epochs,
            warmup_steps=params.warmup_steps,
            per_device_train_batch_size=params.per_device_train_batch_size,
            weight_decay=params.weight_decay,
            logging_steps=params.logging_steps,
            evaluation_strategy=params.evaluation_strategy,
            eval_steps=params.eval_steps,
            save_steps=params.save_steps,
            gradient_accumulation_steps=params.gradient_accumulation_steps
        )
        return model_trainer_config

In [ ]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
import torch

c:\Users\Harsha vardhan\OneDrive\Desktop\TextSummarization-Project\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config=config

    def train(self):
            device="cuda" if torch.cuda.is_available() else "cpu"
            tokenizer=AutoTokenizer.from_pretrained(self.config.model_ckpt)
            model_pegasus=AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
            seq2seq_data_collator=DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

            # loading the dataset
            dataset_samsum_pt=load_from_disk(self.config.data_path)

            trainer_args=TrainingArguments(
                output_dir=self.config.root_dir, num_train_epochs=1, warmup_steps=500,
                per_device_train_batch_size=1, per_device_eval_batch_size=1, weight_decay=0.01, logging_steps=10,
                eval_strategy="steps", eval_steps=500, save_steps=1e6, gradient_accumulation_steps=16
            )
            trainer=Trainer(model=model_pegasus, args=trainer_args,
                    processing_class=tokenizer, data_collator=seq2seq_data_collator,
                    train_dataset=dataset_samsum_pt["train"],
                    eval_dataset=dataset_samsum_pt["validation"]
                    )
            trainer.train()

            ## save model
            model_pegasus.save_pretrained(os.path.join(self.config.root_dir,"pegasus-samsum-model"))
            ## save tokenizer
            tokenizer.save_pretrained(os.path.join(self.config.root_dir,"pegasus-samsum-tokenizer"))

In [ ]:
try:
    config=ConfigurationManager()
    model_trainer_config=config.get_model_trainer_config()
    model_trainer=ModelTrainer(config=model_trainer_config)
    model_trainer.train()
except Exception as e:
    raise e